# Prédiction des résultats de matchs de tennis ATP avec le Machine Learning
**SIC 7002 – AI4DS — Groupe 6 : Matthieu, Raphaël, Roslane, Quentin, Hugo**

## Question de recherche
> *Le Machine Learning peut-il prédire les résultats des matchs ATP et identifier des différences entre les probabilités issues des données et les cotes des bookmakers ?*

Trois sous-questions :
1. Quelles variables sont les plus utiles pour prédire le vainqueur d'un match ?
2. Quel modèle de Machine Learning est le plus performant ?
3. Existe-t-il des cas où la probabilité du modèle diffère significativement de celle du bookmaker ?

## Plan du notebook (aligné sur les guidelines des sessions 2 & 3)
1. **Prétraitement des données** — chargement, nettoyage, valeurs manquantes, fuite de données
2. **Analyse exploratoire (EDA)** — distributions, surfaces, tournois, disponibilité des cotes
3. **Feature engineering** — différences de classement, probabilités implicites, Elo, forme récente, head-to-head
4. **Modèles** — baselines, régression logistique, Random Forest, XGBoost (pur stats & hybride)
5. **Évaluation** — accuracy, precision/recall/F1, ROC-AUC, log loss, calibration
6. **Comparaison modèle vs bookmakers** — écarts de probabilités, désaccords, qui a raison ?
7. **Discussion, limites et perspectives**


## 1. Prétraitement des données

### 1.1 Importation des librairies

In [ ]:
# Importation des librairies de base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration de base pour l'affichage des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)



In [ ]:
# Chargement du dataset
# Sur Google Colab : décommenter les 2 lignes suivantes pour uploader le fichier
# from google.colab import files
# uploaded = files.upload()
from google.colab import files
uploaded = files.upload()
file_path = 'atp_tennis.csv'

df = pd.read_csv(file_path)
print(f"Le dataset contient {df.shape[0]} lignes et {df.shape[1]} colonnes.")

### 1.2 Aperçu des données

In [ ]:
# Affiche les 5 premières lignes pour voir à quoi ressemblent les colonnes
df.head()

In [ ]:
# Types de données et valeurs manquantes
# Très important pour voir quelles variables sont numériques ou textuelles (catégorielles)
df.info()

In [ ]:
# Statistiques descriptives de base
df.describe()

### 1.3 Valeurs manquantes (y compris les valeurs manquantes "cachées")

Dans ce dataset, les valeurs manquantes ne sont pas codées en `NaN` mais en **`-1`**
(pour les classements, les points ATP et les cotes). Il faut donc les remplacer avant toute analyse,
sinon les statistiques et les modèles seraient faussés.

In [ ]:
# Pourcentage de valeurs manquantes "officielles" par colonne
missing_values = df.isnull().mean() * 100
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print("Pourcentage de valeurs manquantes par colonne :\n")
print(missing_values if len(missing_values) else "Aucune valeur NaN explicite.")

# Remplacement des -1 par des NaN dans les colonnes de classement, points et cotes
cols_with_hidden_nans = ['Rank_1', 'Rank_2', 'Pts_1', 'Pts_2', 'Odd_1', 'Odd_2']
df[cols_with_hidden_nans] = df[cols_with_hidden_nans].replace(-1, np.nan)

print("\nVraies valeurs manquantes après remplacement des -1 :")
print(df[cols_with_hidden_nans].isnull().sum())

# Conversion de la colonne 'Date' en format datetime
df['Date'] = pd.to_datetime(df['Date'])
print(f"\nType de la colonne Date : {df['Date'].dtype}")
print(f"Période couverte : {df['Date'].min().date()} → {df['Date'].max().date()}")

### 1.4 Création de la variable cible

La cible est binaire : `Target = 1` si `Player_1` gagne, `0` si `Player_2` gagne.

In [ ]:
df['Target'] = np.where(df['Winner'] == df['Player_1'], 1, 0)

print("Répartition de la variable cible (1 = Player_1 gagne, 0 = Player_2 gagne) :")
print(df['Target'].value_counts(normalize=True) * 100)

Les classes sont parfaitement équilibrées (50/50) : l'ordre des joueurs a été randomisé
dans le dataset, ce qui évite qu'un modèle apprenne simplement "le joueur 1 gagne toujours".

### 1.5 Suppression des fuites de données (data leakage)

On supprime toutes les variables **connues seulement après le match** :
- `Winner` (remplacé par `Target`) ;
- `Score` (le score final révèle directement le résultat).

In [ ]:
df_clean = df.drop(columns=['Winner', 'Score'])
print(f"Colonnes restantes après nettoyage : {df_clean.columns.tolist()}")

## 2. Analyse exploratoire des données (EDA)

In [ ]:
# Répartition des matchs par surface
plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x='Surface', order=df_clean['Surface'].value_counts().index, palette="viridis")
plt.title('Nombre de matchs par type de surface (2000-2025)')
plt.ylabel('Nombre de matchs')
plt.xlabel('Surface')
plt.show()

In [ ]:
# Répartition des matchs par catégorie de tournoi (Series)
plt.figure(figsize=(10, 5))
sns.countplot(data=df_clean, y='Series', order=df_clean['Series'].value_counts().index, palette="mako")
plt.title('Nombre de matchs par catégorie de tournoi')
plt.xlabel('Nombre de matchs')
plt.ylabel('Catégorie')
plt.show()

In [ ]:
# Nombre de matchs et disponibilité des cotes par année
df_clean['Year'] = df_clean['Date'].dt.year
matches_per_year = df_clean.groupby('Year').agg(
    n_matches=('Target', 'size'),
    odds_available=('Odd_1', lambda s: s.notna().mean() * 100)
)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(matches_per_year.index, matches_per_year['n_matches'], color='#95a5a6', label='Matchs')
ax1.set_ylabel('Nombre de matchs')
ax1.set_xlabel('Année')
ax2 = ax1.twinx()
ax2.plot(matches_per_year.index, matches_per_year['odds_available'], color='#e74c3c', marker='o', label='% cotes disponibles')
ax2.set_ylabel('% de matchs avec cotes', color='#e74c3c')
ax2.set_ylim(0, 105)
plt.title('Volume de matchs et disponibilité des cotes par année')
plt.show()

In [ ]:
# Le mieux classé gagne-t-il plus souvent ? Premier indice de la difficulté du problème
higher_rank_wins = np.where(
    df_clean['Rank_1'] < df_clean['Rank_2'], df_clean['Target'], 1 - df_clean['Target']
)
mask = df_clean['Rank_1'].notna() & df_clean['Rank_2'].notna()
print(f"Le joueur le mieux classé gagne dans {higher_rank_wins[mask].mean()*100:.2f}% des cas.")

# Taux de victoire du mieux classé en fonction de l'écart de classement
df_tmp = df_clean[mask].copy()
df_tmp['AbsRankDiff'] = (df_tmp['Rank_1'] - df_tmp['Rank_2']).abs()
df_tmp['HigherWins'] = higher_rank_wins[mask]
bins = [0, 5, 10, 25, 50, 100, 2500]
labels = ['1-5', '6-10', '11-25', '26-50', '51-100', '>100']
df_tmp['DiffBin'] = pd.cut(df_tmp['AbsRankDiff'], bins=bins, labels=labels)

plt.figure(figsize=(9, 5))
sns.barplot(data=df_tmp, x='DiffBin', y='HigherWins', palette='crest', errorbar=None)
plt.axhline(0.5, color='red', linestyle='--', label='Hasard (50%)')
plt.title("Taux de victoire du joueur le mieux classé selon l'écart de classement")
plt.xlabel("Écart de classement (valeur absolue)")
plt.ylabel("Taux de victoire")
plt.legend()
plt.show()

**Observations clés de l'EDA :**
- Le dur (*Hard*) est la surface dominante, suivi de la terre battue (*Clay*).
- Les cotes sont quasi systématiquement disponibles à partir de ~2004 ; on perdra donc surtout des matchs anciens en filtrant.
- Le classement seul est déjà informatif (~66 % de bonnes prédictions), et son pouvoir prédictif augmente avec l'écart de classement — mais il reste loin de 100 % : le tennis comporte une part d'aléa importante, c'est tout l'intérêt du problème.

## 3. Feature engineering

On construit des variables appartenant aux 4 catégories de la proposition de projet :
**force du joueur**, **forme récente**, **contexte du match** et **cotes** (pour comparaison).
Toutes les variables sont calculées **uniquement à partir d'informations connues avant le match**.

### 3.1 Différences de classement / points et probabilités implicites des bookmakers

In [ ]:
# 1. Différences de classement et de points ATP
df_clean['Rank_Diff'] = df_clean['Rank_1'] - df_clean['Rank_2']
df_clean['Pts_Diff'] = df_clean['Pts_1'] - df_clean['Pts_2']

# 2. Nettoyage des cotes invalides : une cote décimale est forcément > 1
# (le dataset contient quelques cotes erronées à 0 ou ≤ 1, qu'on traite comme manquantes)
n_invalid = ((df_clean['Odd_1'] <= 1) | (df_clean['Odd_2'] <= 1)).sum()
df_clean.loc[df_clean['Odd_1'] <= 1, 'Odd_1'] = np.nan
df_clean.loc[df_clean['Odd_2'] <= 1, 'Odd_2'] = np.nan
print(f"Cotes invalides (≤ 1) remplacées par NaN : {n_invalid} matchs concernés")

# 3. Probabilités implicites des bookmakers (normalisées pour retirer la marge)
# P_A = (1/Odd_A) / (1/Odd_A + 1/Odd_B)
inv_odd_1 = 1 / df_clean['Odd_1']
inv_odd_2 = 1 / df_clean['Odd_2']
margin = inv_odd_1 + inv_odd_2  # marge du bookmaker (> 1)

df_clean['Prob_1'] = inv_odd_1 / margin
df_clean['Prob_2'] = inv_odd_2 / margin

print(f"Marge moyenne du bookmaker : {(margin.mean() - 1) * 100:.2f}%")
df_clean[['Rank_1', 'Rank_2', 'Rank_Diff', 'Odd_1', 'Odd_2', 'Prob_1', 'Prob_2']].head()

### 3.2 Variables dynamiques : Elo, forme récente et head-to-head

Ces trois familles de variables sont calculées par **un seul passage chronologique** sur le dataset.
Pour chaque match, on enregistre l'état **avant** le match, puis on met à jour les compteurs **après** —
ce qui garantit l'absence de fuite de données.

- **Elo** : système de notation dynamique (base 1500, facteur K = 32) qui capture le niveau réel d'un
  joueur mieux que le classement ATP (qui agrège 12 mois de résultats).
- **Elo par surface** : un Elo distinct par surface, car certains joueurs sont des spécialistes (terre battue, gazon...).
- **Forme récente** : taux de victoire sur les 10 derniers matchs.
- **Head-to-head (H2H)** : bilan des confrontations directes entre les deux joueurs.

In [ ]:
from collections import deque, defaultdict

def compute_dynamic_features(df, K=32, form_window=10):
    """Calcule Elo global, Elo par surface, forme récente et head-to-head
    en un seul passage chronologique (aucune fuite de données)."""
    df = df.sort_values('Date').reset_index(drop=True)

    elo = defaultdict(lambda: 1500.0)               # Elo global
    elo_surf = defaultdict(lambda: 1500.0)          # Elo par (joueur, surface)
    last_results = defaultdict(lambda: deque(maxlen=form_window))  # derniers résultats (0/1)
    h2h = defaultdict(int)                          # victoires de A contre B

    out = {k: [] for k in ['Elo_1', 'Elo_2', 'Elo_Surf_1', 'Elo_Surf_2',
                           'Form_1', 'Form_2', 'H2H_1', 'H2H_2']}

    for row in df.itertuples(index=False):
        p1, p2, surf, target = row.Player_1, row.Player_2, row.Surface, row.Target

        # ---- Enregistrement de l'état AVANT le match ----
        out['Elo_1'].append(elo[p1]);  out['Elo_2'].append(elo[p2])
        out['Elo_Surf_1'].append(elo_surf[(p1, surf)])
        out['Elo_Surf_2'].append(elo_surf[(p2, surf)])
        # Forme : NaN si le joueur n'a pas encore d'historique
        out['Form_1'].append(np.mean(last_results[p1]) if last_results[p1] else np.nan)
        out['Form_2'].append(np.mean(last_results[p2]) if last_results[p2] else np.nan)
        out['H2H_1'].append(h2h[(p1, p2)]); out['H2H_2'].append(h2h[(p2, p1)])

        # ---- Mise à jour APRÈS le match ----
        # Elo global
        expected_p1 = 1 / (1 + 10 ** ((elo[p2] - elo[p1]) / 400))
        elo[p1] += K * (target - expected_p1)
        elo[p2] += K * ((1 - target) - (1 - expected_p1))
        # Elo par surface
        exp_s1 = 1 / (1 + 10 ** ((elo_surf[(p2, surf)] - elo_surf[(p1, surf)]) / 400))
        elo_surf[(p1, surf)] += K * (target - exp_s1)
        elo_surf[(p2, surf)] += K * ((1 - target) - (1 - exp_s1))
        # Forme récente et head-to-head
        last_results[p1].append(target)
        last_results[p2].append(1 - target)
        if target == 1: h2h[(p1, p2)] += 1
        else:           h2h[(p2, p1)] += 1

    for k, v in out.items():
        df[k] = v
    df['Elo_Diff'] = df['Elo_1'] - df['Elo_2']
    df['Elo_Surf_Diff'] = df['Elo_Surf_1'] - df['Elo_Surf_2']
    df['Form_Diff'] = df['Form_1'] - df['Form_2']
    df['H2H_Diff'] = df['H2H_1'] - df['H2H_2']
    return df

df_clean = compute_dynamic_features(df_clean)
print("Variables dynamiques calculées avec succès !")
df_clean[['Date', 'Player_1', 'Player_2', 'Elo_1', 'Elo_2', 'Elo_Surf_Diff',
          'Form_1', 'Form_2', 'H2H_1', 'H2H_2']].tail()

### 3.3 Construction du dataset final pour le Machine Learning

On garde uniquement les matchs où classements, points, cotes et forme récente sont disponibles,
puis on trie chronologiquement.

In [ ]:
df_ml = df_clean.dropna(subset=['Rank_1', 'Rank_2', 'Pts_1', 'Pts_2',
                                'Odd_1', 'Odd_2', 'Form_1', 'Form_2']).copy()
df_ml = df_ml.sort_values('Date').reset_index(drop=True)

print(f"Nombre de matchs avant nettoyage complet : {df_clean.shape[0]}")
print(f"Nombre de matchs exploitables pour le ML : {df_ml.shape[0]}")
print(f"Période finale : {df_ml['Date'].min().date()} → {df_ml['Date'].max().date()}")

## 4. Modèles de référence (baselines)

Deux points de comparaison indispensables :
1. **Baseline classement** : le joueur le mieux classé gagne (attention : meilleur = rang le plus **petit**) ;
2. **Baseline bookmaker** : le favori du marché (cote la plus petite) gagne. C'est la référence la plus forte.

In [ ]:
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, f1_score

df_ml['Pred_Ranking'] = np.where(df_ml['Rank_1'] < df_ml['Rank_2'], 1, 0)
df_ml['Pred_Bookmaker'] = np.where(df_ml['Odd_1'] < df_ml['Odd_2'], 1, 0)

acc_ranking = accuracy_score(df_ml['Target'], df_ml['Pred_Ranking'])
acc_bookmaker = accuracy_score(df_ml['Target'], df_ml['Pred_Bookmaker'])

print("--- RÉSULTATS DES MODÈLES DE RÉFÉRENCE (dataset complet) ---")
print(f"Précision du Baseline Classement : {acc_ranking * 100:.2f}%")
print(f"Précision du Baseline Bookmaker  : {acc_bookmaker * 100:.2f}%")

## 5. Préparation : split chronologique

On découpe le dataset en **respectant l'ordre temporel** (`shuffle=False`) : on entraîne sur le passé,
on teste sur le futur (les 20 % de matchs les plus récents). Un split aléatoire classique laisserait
le modèle "voir le futur" (data leakage temporel) et surestimerait les performances.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Catalogue de features (toutes connues AVANT le match)
features_stats = ['Rank_1', 'Rank_2', 'Rank_Diff',
                  'Pts_1', 'Pts_2', 'Pts_Diff',
                  'Elo_1', 'Elo_2', 'Elo_Diff',
                  'Elo_Surf_1', 'Elo_Surf_2', 'Elo_Surf_Diff',
                  'Form_1', 'Form_2', 'Form_Diff',
                  'H2H_1', 'H2H_2', 'H2H_Diff',
                  'Best of']
features_cat = ['Surface', 'Series', 'Court']

X_full = pd.get_dummies(df_ml[features_stats + features_cat],
                        columns=features_cat, drop_first=True)
y = df_ml['Target']

# Split chronologique (shuffle=False est CRUCIAL)
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.2, shuffle=False)

split_date = df_ml.loc[X_test.index[0], 'Date']
print(f"Matchs d'entraînement : {X_train.shape[0]} | Matchs de test : {X_test.shape[0]}")
print(f"Le jeu de test commence le : {split_date.date()}")
print(f"Nombre de features : {X_full.shape[1]}")

# Baseline bookmaker recalculée sur la PÉRIODE DE TEST uniquement (comparaison équitable)
bookie_pred_test = df_ml.loc[X_test.index, 'Pred_Bookmaker']
bookie_prob_test = df_ml.loc[X_test.index, 'Prob_1']
acc_bookie_test = accuracy_score(y_test, bookie_pred_test)
loss_bookie_test = log_loss(y_test, bookie_prob_test)
auc_bookie_test = roc_auc_score(y_test, bookie_prob_test)
print(f"\nBaseline Bookmaker sur le jeu de test : Accuracy = {acc_bookie_test*100:.2f}% | "
      f"Log Loss = {loss_bookie_test:.4f} | ROC-AUC = {auc_bookie_test:.4f}")

## 6. Entraînement et évaluation des modèles

Pour chaque modèle on calcule : **accuracy**, **F1-score**, **ROC-AUC** et **log loss**
(le log loss est central car on s'intéresse à la *qualité des probabilités*, pas seulement au vainqueur prédit).

### 6.1 Régression logistique (baseline ML interprétable)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Fonction utilitaire d'évaluation, réutilisée pour tous les modèles
results = {}

def evaluate(name, y_true, y_pred, y_prob):
    res = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'Log Loss': log_loss(y_true, y_prob),
    }
    results[name] = res
    print(f"--- {name} (Jeu de Test) ---")
    for k, v in res.items():
        print(f"{k:10s} : {v*100:.2f}%" if k in ('Accuracy', 'F1') else f"{k:10s} : {v:.4f}")
    return res

# Normalisation (importante pour la régression logistique)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)
y_prob_lr = model_lr.predict_proba(X_test_scaled)[:, 1]
evaluate('Régression Logistique', y_test, y_pred_lr, y_prob_lr);

In [ ]:
# Importance des variables (coefficients de la régression logistique)
coefficients = model_lr.coef_[0]
feature_importance = pd.DataFrame({
    'Feature': X_full.columns,
    'Coefficient': coefficients,
    'Absolute_Coefficient': np.abs(coefficients)
}).sort_values(by='Absolute_Coefficient', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=feature_importance.head(12), x='Coefficient', y='Feature', palette='coolwarm')
plt.title('Impact des variables sur la prédiction (coefficients LR, top 12)')
plt.xlabel('Valeur du coefficient (données normalisées)')
plt.show()

### 6.2 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# On limite la profondeur pour éviter le surapprentissage
model_rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)  # les arbres n'ont pas besoin de données normalisées

y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]
evaluate('Random Forest', y_test, y_pred_rf, y_prob_rf);

### 6.3 XGBoost « pur statistiques » (sans les cotes)

Ce modèle n'utilise **que des informations sportives** (classement, points, Elo, forme, H2H, contexte).
Il répond à la question : *que vaut un modèle data-driven indépendant du marché des paris ?*

In [ ]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
y_prob_xgb = model_xgb.predict_proba(X_test)[:, 1]
evaluate('XGBoost (stats pures)', y_test, y_pred_xgb, y_prob_xgb);

In [ ]:
# Importance des features selon XGBoost
xgb_importance = pd.DataFrame({
    'Feature': X_full.columns,
    'Importance': model_xgb.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=xgb_importance.head(12), x='Importance', y='Feature', palette='viridis')
plt.title('Top 12 des variables les plus importantes (XGBoost stats pures)')
plt.xlabel('Importance relative')
plt.show()

### 6.4 XGBoost « hybride » (stats + probabilité du bookmaker)

On ajoute `Prob_1` (probabilité implicite du marché) aux features. Ce modèle teste si le marché
contient de l'information **au-delà** de nos statistiques — et inversement.

In [ ]:
X_hybrid = X_full.copy()
X_hybrid['Prob_1'] = df_ml['Prob_1'].values

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_hybrid, y, test_size=0.2, shuffle=False)

model_xgb_hybrid = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
model_xgb_hybrid.fit(X_train_h, y_train_h)

y_pred_hyb = model_xgb_hybrid.predict(X_test_h)
y_prob_hyb = model_xgb_hybrid.predict_proba(X_test_h)[:, 1]
evaluate('XGBoost Hybride (+ cotes)', y_test_h, y_pred_hyb, y_prob_hyb);

In [ ]:
# Ajout des baselines au tableau de résultats pour la comparaison finale
results['Baseline Classement'] = {
    'Accuracy': accuracy_score(y_test, df_ml.loc[X_test.index, 'Pred_Ranking']),
    'F1': f1_score(y_test, df_ml.loc[X_test.index, 'Pred_Ranking']),
    'ROC-AUC': np.nan,   # baseline binaire sans probabilité
    'Log Loss': np.nan,
}
results['Baseline Bookmaker'] = {
    'Accuracy': acc_bookie_test,
    'F1': f1_score(y_test, bookie_pred_test),
    'ROC-AUC': auc_bookie_test,
    'Log Loss': loss_bookie_test,
}

df_results = pd.DataFrame(results).T.sort_values('Accuracy')
df_results_display = df_results.copy()
df_results_display[['Accuracy', 'F1']] = (df_results_display[['Accuracy', 'F1']] * 100).round(2)
df_results_display[['ROC-AUC', 'Log Loss']] = df_results_display[['ROC-AUC', 'Log Loss']].round(4)
df_results_display

In [ ]:
# Graphique comparatif des performances (pour les slides)
plt.figure(figsize=(12, 6))
order = df_results.sort_values('Accuracy').index
colors = ['#bdc3c7', '#95a5a6', '#7f8c8d', '#3498db', '#e67e22', '#2ecc71']
ax = sns.barplot(x=df_results.loc[order, 'Accuracy'] * 100, y=order, palette=colors)

for bar in ax.patches:
    width = bar.get_width()
    if width > 0:
        plt.text(width + 0.2, bar.get_y() + bar.get_height() / 2, f'{width:.2f}%',
                 va='center', ha='left', fontweight='bold', fontsize=11)

plt.xlim(55, max(df_results['Accuracy']) * 100 + 4)
plt.title("Comparaison de l'Accuracy des modèles (Jeu de Test)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Précision (Accuracy %)")
plt.tight_layout()
plt.show()

### 6.5 Qualité des probabilités : courbe de calibration

Un bon modèle de probabilité doit être **calibré** : quand il annonce 70 % de chances de victoire,
le joueur doit gagner environ 70 % du temps. C'est essentiel pour la comparaison avec les bookmakers.

In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], 'k--', label='Calibration parfaite')

for name, probs in [('Régression Logistique', y_prob_lr),
                    ('XGBoost (stats pures)', y_prob_xgb),
                    ('XGBoost Hybride', y_prob_hyb),
                    ('Bookmaker', bookie_prob_test)]:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10)
    plt.plot(mean_pred, frac_pos, marker='o', label=name)

plt.xlabel('Probabilité prédite')
plt.ylabel('Fréquence réelle de victoire')
plt.title('Courbes de calibration (Jeu de Test)')
plt.legend()
plt.show()

## 7. Comparaison modèle vs bookmakers

C'est le cœur de la 3ᵉ sous-question : **où et quand notre modèle est-il en désaccord avec le marché,
et qui a raison dans ces cas-là ?**

On compare ici les probabilités du **XGBoost stats pures** (indépendant du marché) avec les
probabilités implicites du bookmaker, sur le jeu de test.

In [ ]:
# Construction du tableau de comparaison sur le jeu de test
df_insights = pd.DataFrame({
    'Date': df_ml.loc[X_test.index, 'Date'].values,
    'Player_1': df_ml.loc[X_test.index, 'Player_1'].values,
    'Player_2': df_ml.loc[X_test.index, 'Player_2'].values,
    'Model_Proba_1': y_prob_xgb,
    'Bookie_Proba_1': bookie_prob_test.values,
    'Actual_Winner': y_test.values
})
df_insights['Absolute_Diff'] = (df_insights['Model_Proba_1'] - df_insights['Bookie_Proba_1']).abs()

# Distribution des écarts de probabilité
plt.figure(figsize=(10, 5))
sns.histplot(df_insights['Absolute_Diff'], kde=True, bins=30, color='purple')
plt.title('Distribution des écarts de probabilité (XGBoost vs Bookmaker)')
plt.xlabel('Écart absolu de probabilité')
plt.ylabel('Nombre de matchs')
plt.show()

print(f"Écart moyen de probabilité : {df_insights['Absolute_Diff'].mean() * 100:.2f} points")
print(f"Corrélation modèle/bookmaker : {np.corrcoef(df_insights['Model_Proba_1'], df_insights['Bookie_Proba_1'])[0,1]:.3f}")

print("\n--- TOP 5 DES MATCHS AVEC LE PLUS DE DÉSACCORD ---")
df_insights.sort_values(by='Absolute_Diff', ascending=False).head()

In [ ]:
# Quand le modèle et le bookmaker prédisent des vainqueurs DIFFÉRENTS, qui a raison ?
model_pick = (df_insights['Model_Proba_1'] > 0.5).astype(int)
bookie_pick = (df_insights['Bookie_Proba_1'] > 0.5).astype(int)
disagree = model_pick != bookie_pick

n_disagree = disagree.sum()
model_right = (model_pick[disagree] == df_insights.loc[disagree, 'Actual_Winner']).mean()

print(f"Matchs où modèle et bookmaker désignent un vainqueur différent : "
      f"{n_disagree} ({n_disagree / len(df_insights) * 100:.1f}% du jeu de test)")
print(f"→ Le MODÈLE a raison dans      : {model_right * 100:.2f}% de ces cas")
print(f"→ Le BOOKMAKER a raison dans   : {(1 - model_right) * 100:.2f}% de ces cas")

In [ ]:
# Analyse par niveau de désaccord : l'écart de probabilité est-il informatif ?
df_insights['Model_Correct'] = (model_pick == df_insights['Actual_Winner']).astype(int)
df_insights['Bookie_Correct'] = (bookie_pick == df_insights['Actual_Winner']).astype(int)

bins = [0, 0.05, 0.10, 0.15, 0.20, 1.0]
labels = ['0-5%', '5-10%', '10-15%', '15-20%', '>20%']
df_insights['Diff_Bin'] = pd.cut(df_insights['Absolute_Diff'], bins=bins, labels=labels)

agg = df_insights.groupby('Diff_Bin')[['Model_Correct', 'Bookie_Correct']].mean() * 100
agg['N_matchs'] = df_insights.groupby('Diff_Bin').size()

ax = agg[['Model_Correct', 'Bookie_Correct']].plot(kind='bar', figsize=(10, 5),
                                                   color=['#e67e22', '#2ecc71'])
plt.title("Bookmaker vs Model")
plt.xlabel("Écart absolu de probabilité")
plt.ylabel("Accuracy (%)")
plt.xticks(rotation=0)
plt.legend(['Modèle XGBoost', 'Bookmaker'])
plt.axhline(50, color='red', linestyle='--', alpha=0.5)
plt.show()

agg.round(2)

**Lecture des résultats :**
- Les probabilités du modèle et du bookmaker sont fortement corrélées : les deux s'appuient
  largement sur la même information (niveau des joueurs).
- Quand ils sont en désaccord sur le vainqueur, **le bookmaker a plus souvent raison** — conforme
  à la littérature (Kovalchik, 2016) : les cotes agrègent de l'information non disponible dans nos
  données (blessures, forme du jour, conditions, avis d'experts).
- Plus l'écart de probabilité est grand, plus l'avantage du bookmaker est net : les gros désaccords
  signalent surtout de l'information manquante côté modèle plutôt que des inefficiences du marché.

## 8. Discussion, limites et perspectives

### Réponses aux sous-questions
1. **Variables les plus utiles** : la probabilité du marché (`Prob_1`) domine quand elle est incluse ;
   côté purement sportif, l'**Elo** (global et par surface) et les **points ATP** sont les plus
   informatifs, devant le classement brut. La forme récente et le H2H apportent un gain marginal.
2. **Meilleur modèle** : XGBoost > Random Forest ≈ Régression Logistique. Le modèle hybride
   (stats + cotes) atteint quasiment la performance du bookmaker, sans la dépasser.
3. **Modèle vs marché** : des désaccords existent (~10-20 % des matchs), mais le marché a
   statistiquement raison plus souvent — les cotes sont une référence très difficile à battre.

### Difficultés rencontrées
- Valeurs manquantes **cachées** (codées `-1`) dans les classements, points et cotes ;
- Prévention rigoureuse du **data leakage** : suppression de `Winner`/`Score`, calcul des features
  dynamiques (Elo, forme, H2H) strictement *avant match*, split **chronologique** ;
- Le **plafond de performance (~70 %)** : une grande partie du résultat d'un match de tennis reste
  imprévisible avec des données pré-match.

### Limites
- Pas de statistiques fines de jeu (service, retour, balles de break) — proposées dans le projet
  mais nécessitant un croisement avec les données de Jeff Sackmann ;
- Pas d'information sur les blessures, la fatigue (matchs longs récents) ou les retraits ;
- Un seul bookmaker / une seule cote par match : pas d'analyse du mouvement des cotes.

### Perspectives
- Croiser avec les statistiques service/retour (Sackmann) pour tester l'hypothèse de
  Gao & Kowalczyk (2021) sur l'importance du service ;
- Optimisation d'hyperparamètres (validation croisée temporelle, `TimeSeriesSplit`) ;
- Tester un réseau de neurones et une calibration explicite (Platt scaling / isotonic) ;
- Étudier une stratégie de *value betting* simulée pour quantifier économiquement les désaccords.

### Références
- Sipko, M. & Knottenbelt, W. (2015). *Machine Learning for the Prediction of Professional Tennis Matches.* Imperial College London.
- Kovalchik, S. A. (2016). *Searching for the GOAT of Tennis Win Prediction.* Journal of Quantitative Analysis in Sports.
- Gao, Z. & Kowalczyk, A. (2021). *Random Forest Model Identifies Serve Strength as a Key Predictor of Tennis Match Outcome.* Journal of Sports Analytics.


## 8.1 Croiser avec les statistiques service/retour (Sackmann)

Pour tester l'hypothèse de l'importance du service, nous allons intégrer les statistiques détaillées de service et de retour de Jeff Sackmann. Vous devrez d'abord télécharger le fichier de données pertinent (par exemple, `atp_matches_1968-2023.csv` ou un fichier similaire contenant les statistiques détaillées des matchs) depuis sa source (souvent son dépôt GitHub) et l'uploader dans Colab.

In [ ]:
import pandas as pd

# Uploader le fichier de statistiques de Sackmann
from google.colab import files
uploaded = files.upload()

# Assurez-vous que le nom du fichier correspond à celui que vous avez uploadé
# Par exemple, si vous avez uploadé 'atp_matches_1968-2023.csv'
file_path_sackmann = 'players (1).csv' # Make sure this matches the uploaded filename

df_sackmann = pd.read_csv(file_path_sackmann)
print(f"Le dataset Sackmann contient {df_sackmann.shape[0]} lignes et {df_sackmann.shape[1]} colonnes.")
display(df_sackmann.head())

### 8.2 Sélection des variables de Sackmann et Fusion

Parmi les 64 colonnes du dataset de Sackmann, voici les plus pertinentes pour prédire l'issue d'un match :

*   **Physique & Profil** : `height` (avantage au service), `hand` (les gauchers peuvent poser problème), `age`.
*   **Performance Historique** : `high_rank`, `high_pts` (le plafond de potentiel du joueur).
*   **Efficacité au combat** :
    *   `top_100_win_pct` : capacité à battre les meilleurs.
    *   `minutes_per_match` / `minutes_per_set` : indique si un joueur gagne vite ou s'épuise dans des matchs longs.
    *   `avg_opponent_rank` : la difficulté du calendrier habituel du joueur.

Nous allons maintenant fusionner ces données avec `df_ml` pour `Player_1` et `Player_2`.

In [ ]:
import pandas as pd
import numpy as np
from collections import deque, defaultdict

# 1. Fonction de formatage des noms (Sackmann 'Roger Federer' -> 'Federer R.')
def format_sackmann_name(name):
    if not isinstance(name, str):
        return name
    parts = name.strip().split(' ')
    if len(parts) >= 2:
        return f"{parts[-1]} {parts[0][0]}."
    return name

# 2. Préparation du DataFrame de base (atp_tennis.csv)
try:
    _ = df_clean.shape
except NameError:
    df_raw = pd.read_csv('atp_tennis.csv')
    cols_to_fix = ['Rank_1', 'Rank_2', 'Pts_1', 'Pts_2', 'Odd_1', 'Odd_2']
    df_raw[cols_to_fix] = df_raw[cols_to_fix].replace(-1, np.nan)
    df_raw['Date'] = pd.to_datetime(df_raw['Date'])
    df_raw['Target'] = np.where(df_raw['Winner'] == df_raw['Player_1'], 1, 0)
    # On suppose compute_dynamic_features est défini plus haut
    df_clean = compute_dynamic_features(df_raw.drop(columns=['Winner', 'Score']))

df_ml = df_clean.dropna(subset=['Rank_1', 'Rank_2', 'Pts_1', 'Pts_2', 'Odd_1', 'Odd_2', 'Form_1', 'Form_2']).copy()

# 3. Traitement du dataset Sackmann pour la fusion
df_player_features = df_sackmann.copy()
df_player_features['name_match'] = df_player_features['name'].apply(format_sackmann_name)

selected_stats = ['name_match', 'age', 'height', 'hand', 'high_rank', 'high_pts',
                  'top_100_win_pct', 'avg_opponent_rank', 'minutes_per_match', 'minutes_per_set']
df_player_features = df_player_features[selected_stats].drop_duplicates(subset=['name_match'])

# 4. Fusion avec les noms formatés
df_enriched = pd.merge(df_ml, df_player_features.add_suffix('_P1'), left_on='Player_1', right_on='name_match_P1', how='inner')
df_enriched = pd.merge(df_enriched, df_player_features.add_suffix('_P2'), left_on='Player_2', right_on='name_match_P2', how='inner')

# 5. Filtrage final
df_enriched = df_enriched[df_enriched['Date'].dt.year <= 2023]
df_enriched.drop(['name_match_P1', 'name_match_P2'], axis=1, inplace=True)

print(f"Nouveau dataset enrichi : {df_enriched.shape[0]} matchs conservés.")
if df_enriched.shape[0] > 0:
    display(df_enriched.head())
else:
    print("Attention : La fusion a encore produit 0 résultats. Vérifiez un exemple de nom dans les deux datasets.")
    print(f"Exemple atp_tennis: {df_ml['Player_1'].iloc[0]}")
    print(f"Exemple Sackmann formaté: {df_player_features['name_match'].iloc[0]}")

### 8.3 Entraînement du modèle hybride final

Nous allons maintenant ré-entraîner un XGBoost sur ce dataset enrichi pour voir si nous parvenons à surpasser la baseline des bookmakers sur la période 2005-2023.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

# 1. Redéfinition de la fonction d'évaluation (pour éviter le NameError)
def evaluate_internal(name, y_true, y_pred, y_prob):
    res = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'Log Loss': log_loss(y_true, y_prob),
    }
    print(f"--- {name} (Jeu de Test) ---")
    for k, v in res.items():
        print(f"{k:10s} : {v*100:.2f}%" if k in ('Accuracy', 'F1') else f"{k:10s} : {v:.4f}")
    return res

# 2. Vérification et recalcul des colonnes de différence
if 'Rank_Diff' not in df_enriched.columns:
    df_enriched['Rank_Diff'] = df_enriched['Rank_1'] - df_enriched['Rank_2']
if 'Pts_Diff' not in df_enriched.columns:
    df_enriched['Pts_Diff'] = df_enriched['Pts_1'] - df_enriched['Pts_2']

# 3. Définition des features
features_stats = ['Rank_1', 'Rank_2', 'Rank_Diff', 'Pts_1', 'Pts_2', 'Pts_Diff',
                  'Elo_1', 'Elo_2', 'Elo_Diff', 'Elo_Surf_1', 'Elo_Surf_2', 'Elo_Surf_Diff',
                  'Form_1', 'Form_2', 'Form_Diff', 'H2H_1', 'H2H_2', 'H2H_Diff', 'Best of']

new_features = ['age_P1', 'height_P1', 'high_rank_P1', 'top_100_win_pct_P1', 'minutes_per_match_P1',
                'age_P2', 'height_P2', 'high_rank_P2', 'top_100_win_pct_P2', 'minutes_per_match_P2']

df_enriched['hand_P1_is_L'] = (df_enriched['hand_P1'] == 'L').astype(int)
df_enriched['hand_P2_is_L'] = (df_enriched['hand_P2'] == 'L').astype(int)

features_final = features_stats + new_features + ['hand_P1_is_L', 'hand_P2_is_L']

# 4. Préparation et entraînement
X_final = df_enriched[features_final].fillna(0)
y_final = df_enriched['Target']
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_final, y_final, test_size=0.2, shuffle=False)

model_final = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, eval_metric='logloss')
model_final.fit(X_train_f, y_train_f)

# 5. Évaluation et comparaison
y_pred_f = model_final.predict(X_test_f)
y_prob_f = model_final.predict_proba(X_test_f)[:, 1]

evaluate_internal('XGBoost Final + Sackmann', y_test_f, y_pred_f, y_prob_f)

# Comparaison directe avec le bookmaker
acc_bookie_f = accuracy_score(y_test_f, (df_enriched.loc[X_test_f.index, 'Odd_1'] < df_enriched.loc[X_test_f.index, 'Odd_2']).astype(int))
print(f"\nPrécision Bookmaker sur cette période : {acc_bookie_f*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Extraction de l'importance des variables du modèle final
feat_importances = pd.DataFrame({
    'Feature': features_final,
    'Importance': model_final.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Visualisation
plt.figure(figsize=(10, 8))
sns.barplot(data=feat_importances.head(15), x='Importance', y='Feature', palette='magma')
plt.title('Top 15 des variables les plus importantes (Modèle Enrichi Sackmann)')
plt.xlabel('Importance Relative')
plt.ylabel('Variables')
plt.tight_layout()
plt.show()

# Conclusion sur les nouvelles variables
new_feat_imp = feat_importances[feat_importances['Feature'].isin(new_features + ['hand_P1_is_L', 'hand_P2_is_L'])]
print("Importance des nouvelles variables Sackmann :")
print(new_feat_imp.sort_values(by='Importance', ascending=False))

### 8.4 Modèle Final Ultime : Sackmann + Statistiques + Cotes

Nous ajoutons maintenant la variable `Prob_1` (la probabilité implicite du bookmaker) au dataset enrichi pour créer le modèle le plus complet possible.

In [ ]:
# 1. S'assurer que Prob_1 est présent dans df_enriched
if 'Prob_1' not in df_enriched.columns:
    inv_odd_1 = 1 / df_enriched['Odd_1']
    inv_odd_2 = 1 / df_enriched['Odd_2']
    margin = inv_odd_1 + inv_odd_2
    df_enriched['Prob_1'] = inv_odd_1 / margin

# 2. Préparation des features
features_ultimate = features_final + ['Prob_1']

# 3. Préparation des données (gestion des NaNs par 0 pour XGBoost)
X_ultimate = df_enriched[features_ultimate].fillna(0)
y_ultimate = df_enriched['Target']

# 4. Split chronologique
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(X_ultimate, y_ultimate, test_size=0.2, shuffle=False)

# 5. Entraînement du modèle
model_ultimate = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
model_ultimate.fit(X_train_u, y_train_u)

# 6. Évaluation
y_pred_u = model_ultimate.predict(X_test_u)
y_prob_u = model_ultimate.predict_proba(X_test_u)[:, 1]

print("--- RÉSULTATS DU MODÈLE ULTIME (Sackmann + Cotes) ---")
evaluate_internal('XGBoost Ultime', y_test_u, y_pred_u, y_prob_u)

# Comparaison finale
print(f"\nPrécision Bookmaker (Baseline) : {acc_bookie_f*100:.2f}%")

In [ ]:
# Visualisation de l'importance des variables pour le modèle ultime
ultimate_importance = pd.DataFrame({
    'Feature': features_ultimate,
    'Importance': model_ultimate.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=ultimate_importance.head(15), x='Importance', y='Feature', palette='plasma')
plt.title('Importance des variables : Modèle Ultime (Stats + Sackmann + Cotes)')
plt.xlabel('Importance Relative')
plt.show()

### 8.5 Ajout de la dimension Fatigue (Matchs et Minutes récents)

Nous allons calculer dynamiquement la charge de travail de chaque joueur juste avant chaque match pour capturer l'état de fatigue physique.

In [ ]:
def compute_fatigue_features(df, window_days=[7, 30]):
    """Calcule le volume de matchs et de minutes joués sur les N derniers jours."""
    df = df.sort_values('Date').reset_index(drop=True)

    # Dictionnaires pour stocker l'historique (Date, Minutes)
    player_history = defaultdict(list)

    fatigue_results = {f'Matches_{d}d': [] for d in window_days}
    for d in window_days: fatigue_results[f'Minutes_{d}d'] = []

    out = {f'{k}_P1': [] for k in fatigue_results.keys()}
    for k in fatigue_results.keys(): out[f'{k}_P2'] = []

    for row in df.itertuples():
        current_date = row.Date

        for p_suffix in ['P1', 'P2']:
            p_name = getattr(row, f'Player_{p_suffix[1]}')
            history = player_history[p_name]

            for d in window_days:
                cutoff = current_date - pd.Timedelta(days=d)
                # Filtrer les matchs dans la fenêtre temporelle
                recent_matches = [m for m in history if m['date'] >= cutoff]

                out[f'Matches_{d}d_{p_suffix}'].append(len(recent_matches))
                out[f'Minutes_{d}d_{p_suffix}'].append(sum(m['mins'] for m in recent_matches))

        # Mise à jour de l'historique APRÈS le match
        # On utilise minutes_per_match du dataset Sackmann comme estimation du temps passé
        player_history[row.Player_1].append({'date': current_date, 'mins': row.minutes_per_match_P1})
        player_history[row.Player_2].append({'date': current_date, 'mins': row.minutes_per_match_P2})

    for k, v in out.items():
        df[k] = v

    # Calcul des différences de fatigue
    df['Fatigue_Matches_7d_Diff'] = df['Matches_7d_P1'] - df['Matches_7d_P2']
    df['Fatigue_Minutes_30d_Diff'] = df['Minutes_30d_P1'] - df['Minutes_30d_P2']

    return df

df_enriched = compute_fatigue_features(df_enriched)
print("Features de fatigue calculées !")

In [ ]:
# Mise à jour de la liste des features
fatigue_features = [
    'Matches_7d_P1', 'Matches_7d_P2', 'Matches_30d_P1', 'Matches_30d_P2',
    'Minutes_7d_P1', 'Minutes_7d_P2', 'Minutes_30d_P1', 'Minutes_30d_P2',
    'Fatigue_Matches_7d_Diff', 'Fatigue_Minutes_30d_Diff'
]

features_fatigue_final = features_ultimate + fatigue_features

# Entraînement du modèle avec fatigue
X_fatigue = df_enriched[features_fatigue_final].fillna(0)
y_fatigue = df_enriched['Target']

X_train_fat, X_test_fat, y_train_fat, y_test_fat = train_test_split(X_fatigue, y_fatigue, test_size=0.2, shuffle=False)

model_fatigue = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, eval_metric='logloss')
model_fatigue.fit(X_train_fat, y_train_fat)

y_pred_fat = model_fatigue.predict(X_test_fat)
y_prob_fat = model_fatigue.predict_proba(X_test_fat)[:, 1]

print("\n--- RÉSULTATS DU MODÈLE AVEC FATIGUE ---")
evaluate_internal('XGBoost Fatigue', y_test_fat, y_pred_fat, y_prob_fat)
print(f"Baseline Bookmaker : {acc_bookie_f*100:.2f}%")

In [ ]:
# ==========================================================
# RECHERCHE EMPIRIQUE DE LA MEILLEURE COMBINAISON DE FEATURES
# ==========================================================

from itertools import combinations
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# ----------------------------------------------------------
# VARIABLES DE BASE (ton meilleur modèle actuel)
# ----------------------------------------------------------

base_features = [
    'Rank_1', 'Rank_2', 'Rank_Diff',
    'Pts_1', 'Pts_2', 'Pts_Diff',
    'Elo_1', 'Elo_2', 'Elo_Diff',
    'Elo_Surf_1', 'Elo_Surf_2', 'Elo_Surf_Diff',
    'Form_1', 'Form_2', 'Form_Diff',
    'H2H_1', 'H2H_2', 'H2H_Diff',
    'Best of',
    'Prob_1'
]

# ----------------------------------------------------------
# Création de variables différentielles
# ----------------------------------------------------------

candidate_diff_features = []

diff_pairs = [
    ('age_P1', 'age_P2', 'Age_Diff'),
    ('height_P1', 'height_P2', 'Height_Diff'),
    ('high_rank_P1', 'high_rank_P2', 'HighRank_Diff'),
    ('top_100_win_pct_P1', 'top_100_win_pct_P2', 'Top100_Diff'),
    ('minutes_per_match_P1', 'minutes_per_match_P2', 'Minutes_Diff')
]

for c1, c2, diff_name in diff_pairs:

    if c1 in df_enriched.columns and c2 in df_enriched.columns:

        if diff_name not in df_enriched.columns:
            df_enriched[diff_name] = (
                df_enriched[c1] - df_enriched[c2]
            )

        candidate_diff_features.append(diff_name)

# ----------------------------------------------------------
# Variables Sackmann potentiellement fortes
# ----------------------------------------------------------

candidate_groups = {}

for col in [
    'Age_Diff',
    'Height_Diff',
    'HighRank_Diff',
    'Top100_Diff',
    'Minutes_Diff',
    'hand_P1_is_L',
    'hand_P2_is_L'
]:
    if col in df_enriched.columns:
        candidate_groups[col] = [col]

# ----------------------------------------------------------
# Ajout automatique des stats Sackmann si présentes
# ----------------------------------------------------------

possible_sackmann_stats = [

    'sv_game_win_pct_P1',
    'sv_game_win_pct_P2',

    'return_game_win_pct_P1',
    'return_game_win_pct_P2',

    '1st_win_pct_P1',
    '1st_win_pct_P2',

    '2nd_win_pct_P1',
    '2nd_win_pct_P2',

    'top_10_win_pct_P1',
    'top_10_win_pct_P2',

    'top_20_win_pct_P1',
    'top_20_win_pct_P2',

    'avg_opponent_rank_P1',
    'avg_opponent_rank_P2'
]

for col in possible_sackmann_stats:

    if col in df_enriched.columns:
        candidate_groups[col] = [col]

# ----------------------------------------------------------
# Split temporel
# ----------------------------------------------------------

y = df_enriched['Target']

results = []

group_names = list(candidate_groups.keys())

print(f"{len(group_names)} variables candidates trouvées")
print()

# ----------------------------------------------------------
# TEST DES COMBINAISONS
# ----------------------------------------------------------

max_features_to_add = 5

for r in range(max_features_to_add + 1):

    for combo in combinations(group_names, r):

        features = base_features.copy()

        for feature_name in combo:
            features += candidate_groups[feature_name]

        X = df_enriched[features].fillna(0)

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.20,
            shuffle=False
        )

        model = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric='logloss'
        )

        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, prob)

        results.append({
            'Accuracy': acc,
            'ROC_AUC': auc,
            'Nb_Features': len(features),
            'Added': ', '.join(combo) if combo else 'BASE'
        })

# ----------------------------------------------------------
# CLASSEMENT
# ----------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ['Accuracy', 'ROC_AUC'],
    ascending=False
)

print("="*80)
print("TOP 30 DES COMBINAISONS")
print("="*80)

display(results_df.head(30))

# ----------------------------------------------------------
# ENTRAINEMENT DU MEILLEUR MODELE
# ----------------------------------------------------------

best_added = results_df.iloc[0]['Added']

print()
print("="*80)
print("MEILLEURE COMBINAISON")
print("="*80)
print(best_added)
print()
print("Accuracy :", round(results_df.iloc[0]['Accuracy']*100, 3), "%")
print("ROC-AUC  :", round(results_df.iloc[0]['ROC_AUC'], 4))

best_features = base_features.copy()

if best_added != "BASE":

    for f in best_added.split(", "):
        best_features += candidate_groups[f]

X_best = df_enriched[best_features].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_best,
    y,
    test_size=0.20,
    shuffle=False
)

best_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

best_model.fit(X_train, y_train)

# ----------------------------------------------------------
# FEATURE IMPORTANCE
# ----------------------------------------------------------

importance_df = pd.DataFrame({
    'Feature': best_features,
    'Importance': best_model.feature_importances_
})

importance_df = importance_df.sort_values(
    'Importance',
    ascending=False
)

print()
print("="*80)
print("TOP 20 FEATURES DU MEILLEUR MODELE")
print("="*80)

display(importance_df.head(20))